In [ ]:
import pandas as pd
import requests
from pykrx import stock

# 1) KRX “엑셀” 다운로드 (실제론 HTML 테이블)
url  = "http://kind.krx.co.kr/corpgeneral/corpList.do?method=download&searchType=13"
resp = requests.get(url)
resp.raise_for_status()

# 2) HTML 테이블 파싱 (첫 번째 테이블 사용)
tables = pd.read_html(resp.text, header=0)
df = tables[0]

# 3) 컬럼 추출 및 타입 변환
df = df[["종목코드", "회사명", "업종"]]
df.columns = ["ticker", "company_name", "industry"]

# 종목코드를 6자리 문자열로 zero-fill
df["ticker"] = df["ticker"].astype(str).str.zfill(6)

# 4) 코스피200 종목만 필터링
kospi200 = stock.get_index_portfolio_deposit_file("1028")
kospi200_df = pd.DataFrame({"ticker": kospi200})

merged = kospi200_df.merge(df, on="ticker", how="left")

print(merged.head())
